# Parallel Execution

*Notebook 16*

Run multiple agents simultaneously to cut latency when tasks don't depend on each other.
<br>
<br>
**Topics:**
- Sequential vs parallel execution — when each makes sense

- Running agents concurrently with `asyncio.gather()`

- Merging results from parallel agents

- Cost and speed tradeoffs

- Handling failed tasks with `return_exceptions=True`

- Partial result handling

- Timeout and fallback behavior

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner

# Notebook-specific imports
import asyncio
import time

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

When tasks are independent, running agents sequentially wastes time.

Parallel execution runs all tasks at once — three agents that each take 5 seconds finish in 5 seconds, not 15.

---

## ⚡ Async Recap

Two syntax patterns power everything in this notebook:

`await` — pauses until one coroutine finishes before moving to the next line.

`asyncio.gather()` — launches multiple async tasks (called coroutines) at the same time and waits for all of them.

If any coroutine inside `gather()` raises an exception, the default behavior is to propagate it immediately — Part 6 shows how to handle this.

---

## ⏱️ Part 1: Sequential vs Parallel — The Difference

**Sequential:** each agent waits for the previous to finish.
```
Agent A ──────────────────►
                            Agent B ──────────────────►
                                                       Agent C ──►
Total time: A + B + C
```

**Parallel:** all agents start at the same time.
```
Agent A ──────────────────►
Agent B ──────────────────►
Agent C ──────────────────►
Total time: max(A, B, C)
```

Use parallel when tasks are **independent** — no agent needs another's output to start.

---

## 🐢 Part 2: Sequential Baseline

First, let's see how long it takes to run three independent analyses sequentially.

In [ ]:
# Three independent analysis agents
market_agent = Agent(
    name="MarketAnalyst",
    instructions="You analyze market trends and opportunities. Be specific and concise.",
    model=MODEL
)

tech_agent = Agent(
    name="TechAnalyst",
    instructions="You analyze technical feasibility and implementation challenges. Be specific and concise.",
    model=MODEL
)

risk_agent = Agent(
    name="RiskAnalyst",
    instructions="You analyze risks and potential failure modes. Be specific and concise.",
    model=MODEL
)

topic = "Building an AI-powered personal finance app"

# --------------------------------------------------------------
print("✅ Three analysis agents ready")

#### Run Sequentially

In [ ]:
print("🐢 SEQUENTIAL EXECUTION")
print("=" * 60)

start = time.time()

market_result = await Runner.run(market_agent, input=topic)

tech_result = await Runner.run(tech_agent, input=topic)

risk_result = await Runner.run(risk_agent, input=topic)

sequential_time = time.time() - start

print(f"⏱️  Total time: {sequential_time:.1f}s")
print(f"\n📊 Market: {market_result.final_output}")
print(f"\n⚙️  Tech:   {tech_result.final_output}")
print(f"\n⚠️  Risk:   {risk_result.final_output}")
print("=" * 60)

---

## ⚡ Part 3: Parallel with `asyncio.gather()`

`asyncio.gather()` runs multiple coroutines concurrently and returns their results in the same order they were passed in.

In your own code, the conversion is usually simple: take several independent `await Runner.run(...)` calls and place those same coroutines inside one `await asyncio.gather(...)` call instead.

In [ ]:
print("⚡ PARALLEL EXECUTION")
print("=" * 60)

start = time.time()

market_result, tech_result, risk_result = await asyncio.gather(
    Runner.run(market_agent, input=topic),
    Runner.run(tech_agent, input=topic),
    Runner.run(risk_agent, input=topic)
)

parallel_time = time.time() - start

print(f"⏱️  Total time: {parallel_time:.1f}s")
print(f"📉 Speedup: {sequential_time / parallel_time:.1f}x faster")
print(f"\n📊 Market: {market_result.final_output}")
print(f"\n⚙️  Tech:   {tech_result.final_output}")
print(f"\n⚠️  Risk:   {risk_result.final_output}")
print("=" * 60)

---

## 🔀 Part 4: Parallel + Merge

The real pattern: run specialists in parallel, then merge their outputs into one final response with a synthesis agent.

This is the **fan-out / fan-in** pattern — fan out to launch specialists in parallel, fan in to merge their outputs.

In [ ]:
synthesis_instructions = (
    "You synthesize multiple expert analyses into a clear executive summary.\n"
    "Present the key points from each analysis and end with an overall recommendation."
)

synthesis_agent = Agent(
    name="SynthesisAgent",
    instructions=synthesis_instructions,
    model=MODEL
)

# --------------------------------------------------------------
print("✅ Synthesis agent ready")

#### Run Parallel + Synthesis

In [ ]:
print("🔀 PARALLEL + MERGE PIPELINE")
print("=" * 60)
print(f"Topic: {topic}\n")

start = time.time()

# Phase 1: Run all specialists in parallel
print("Phase 1: Running specialists in parallel...")

market_result, tech_result, risk_result = await asyncio.gather(
    Runner.run(market_agent, input=topic),
    Runner.run(tech_agent, input=topic),
    Runner.run(risk_agent, input=topic)
)

print(f"  ✓ All specialists complete ({time.time() - start:.1f}s)\n")

# Phase 2: Synthesize results
print("Phase 2: Synthesizing results...")
synthesis_input = (
    f"Synthesize these expert analyses on: {topic}\n\n"
    f"MARKET ANALYSIS:\n{market_result.final_output}\n\n"
    f"TECHNICAL ANALYSIS:\n{tech_result.final_output}\n\n"
    f"RISK ANALYSIS:\n{risk_result.final_output}"
)

synthesis_result = await Runner.run(synthesis_agent, input=synthesis_input)

total_time = time.time() - start

print(f"  ✓ Synthesis complete ({total_time:.1f}s total)")
print("\n" + "=" * 60)
print("📋 EXECUTIVE SUMMARY")
print("=" * 60)
print(synthesis_result.final_output)

### 💡 Why This Works

Fan-out reduces total latency to the slowest specialist — all tasks run simultaneously rather than in sequence.

The synthesis agent merges independent perspectives into a unified output.

---

## 💰 Part 5: Cost & Speed Tradeoffs

Parallel execution is faster but not cheaper — all agents still make API calls and consume tokens.

Parallel wins on latency when tasks are independent, but it does not save money and can increase pressure on rate limits.

One exception: running several mini-model agents in parallel can be faster and cheaper than a single reasoning-model call for the same work.

<div style="text-align: left; display: inline-block;">

| | Sequential | Parallel |
|---|---|---|
| Latency | A + B + C | max(A, B, C) |
| Token cost | Same | Same |
| API calls | Same | Same |
| Use when | Tasks depend on each other | Tasks are independent |

</div>

---

## ⚠️ Part 6: When One Task Fails

By default `asyncio.gather()` raises immediately when any task fails — the exception propagates and you lose all results, including the ones that succeeded.

The fix is one argument: `return_exceptions=True`.

In [ ]:
# ❌ Default behavior — one failure brings down the whole gather
async def analyze_topic(topic):
    agent = Agent(
        name="Analyst",
        instructions="Write two sentences analyzing this topic.",
        model=MODEL
    )

    result = await Runner.run(agent, input=f"Analyze: {topic}")

    return result.final_output

async def bad_task(topic):
    if topic == "SKIP":
        raise ValueError(f"Skipping topic: {topic}")
    return await analyze_topic(topic)

# --------------------------------------------------------------
topics = ["Python", "SKIP", "JavaScript"]
try:
    results = await asyncio.gather(*[bad_task(t) for t in topics])
    print(results)
except Exception as e:
    print(f"❌ Entire gather failed: {e}")
    print("   Python and JavaScript results are lost")

### The Fix: `return_exceptions=True`

Exceptions are returned as values in the results list instead of raised — you keep all results and handle failures explicitly.

In [ ]:
# ✅ return_exceptions=True — each result is either output or an Exception
results = await asyncio.gather(
    bad_task("Python"),
    bad_task("SKIP"),
    bad_task("JavaScript"),
    return_exceptions=True
)

for topic, result in zip(topics, results):
    if isinstance(result, Exception):
        print(f"⚠️  '{topic}' failed: {result}")
    else:
        print(f"✅ '{topic}': {result[:80]}")

### Partial Result Handling

With partial results you have three options depending on how critical the failed task is.

In [ ]:
# Using `results` from the previous cell, `topics` from the first demo

# Pattern: collect what succeeded, decide how to proceed
good = [(t, r) for t, r in zip(topics, results) if not isinstance(r, Exception)]
failed = [t for t, r in zip(topics, results) if isinstance(r, Exception)]

print(f"✅ {len(good)} succeeded, ⚠️  {len(failed)} failed")

if len(good) == 0:
    print("❌ All tasks failed — abort")
elif len(good) < 2:
    print("⚠️  Too few results — consider retrying failed tasks")
else:
    print("✅ Enough results — proceed with synthesis")
    combined = "\n\n".join(f"{t}: {r}" for t, r in good)

    partial_synthesis_agent = Agent(
        name="Synthesizer",
        instructions="Combine the provided analyses into a brief summary. Note any gaps.",
        model=MODEL
    )

    result = await Runner.run(partial_synthesis_agent, input=f"Synthesize (note: {failed} topics were unavailable):\n{combined}")

    print(f"\nSynthesis:\n{result.final_output[:300]}")

# In your own pipeline, replace the artificial `bad_task` list with
# `asyncio.gather(...)` of your real specialist agents from Part 4, and set
# the threshold based on which results are critical — e.g., abort if a
# required specialist failed, proceed if at least one of each category
# succeeded.

### 💡 Why This Works

`return_exceptions=True` converts exceptions from "crash the gather" to "return alongside results" — the `isinstance(result, Exception)` check then separates good from failures.

This pattern prevents silent degradation — you always know exactly which tasks failed and why.

---

## ⏱️ Part 7: Timeouts and Fallbacks

`return_exceptions=True` handles errors, but a hung or slow agent can still block the whole `gather()`. Timeouts give every task a deadline.

In [ ]:
async def run_with_timeout(agent, message, timeout_seconds=10):
    """Run an agent with a timeout. Returns output or None if timed out."""
    try:
        result = await asyncio.wait_for(

            Runner.run(agent, input=message),

            timeout=timeout_seconds
        )
        return result.final_output
    except asyncio.TimeoutError:
        return None

# --------------------------------------------------------------
print("✅ run_with_timeout() ready")

#### Run with Timeout

In [ ]:
slow_agent = Agent(
    name="SlowAgent",
    instructions="You are a helpful assistant.",
    model=MODEL
)

# Call 1: Normal timeout — completes before the deadline
print("Call 1: 15-second timeout")
print("-" * 60)
timeout_result = await run_with_timeout(slow_agent, "What is 2 + 2?", timeout_seconds=15)

if timeout_result is None:
    print("⚠️ Agent timed out — using fallback response")
    fallback = "Analysis unavailable due to timeout."
    print(f"Fallback: {fallback}")
else:
    print(f"✅ Result: {timeout_result}")

# Call 2: Forced timeout — deadline too short for any API call to finish
print("\nCall 2: 0.001-second timeout (forced failure)")
print("-" * 60)
timeout_result = await run_with_timeout(slow_agent, "What is 2 + 2?", timeout_seconds=0.001)

if timeout_result is None:
    print("⚠️ Agent timed out — using fallback response")
    fallback = "Analysis unavailable due to timeout."
    print(f"Fallback: {fallback}")
else:
    print(f"✅ Result: {timeout_result}")

### 💡 Why This Works

`asyncio.wait_for()` wraps any coroutine with a deadline.

When the deadline passes it raises `asyncio.TimeoutError` — catching it lets you return a fallback instead of crashing.

Always decide explicitly: skip the failed task, use a default, or abort the workflow.

---

### Exercise 1: Parallel Translation

*Covers: `asyncio.gather` — running agents in parallel*

Translate a piece of text into three languages simultaneously.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Parallel Translation
# --------------------------------------------------------------
# Objective: Translate the same text to three languages in parallel.

text = "Artificial intelligence is changing how we build software."

# TODO 1: Create three translation agents — Spanish, French, Japanese
#           Each agent's instructions: translate text to [language] only

# TODO 2: Run all three in parallel using asyncio.gather()

# TODO 3: Print each translation with its language label
#           and the total elapsed time

# --- Write your code below this line ---

### Exercise 2: Parallel Research + Synthesis

*Covers: `asyncio.gather` — parallel research with synthesis*

Research a topic from three different angles simultaneously, then merge the results.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Parallel Research + Synthesis
# --------------------------------------------------------------
# Objective: Three researchers run in parallel, one synthesizer merges.

research_topic = "The rise of remote work since 2020"

# TODO 1: Create three researcher agents with different lenses:
#           - Economic impact
#           - Social and cultural effects
#           - Technology and tools

# TODO 2: Run all three in parallel

# TODO 3: Create a synthesis agent that combines all three perspectives
#           into a structured report

# TODO 4: Run the synthesis agent with the combined parallel results
#           Print the final report and total elapsed time

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**`asyncio.gather()` is the mechanism:**

- Pass multiple `Runner.run()` coroutines — they all start at the same time

- Results come back in the same order as the inputs
<br>
<br>

**Use parallel when tasks are independent:**

- No agent needs another's output to start

- Same cost as sequential — just faster
<br>
<br>

**Fan-out / Fan-in is the full pattern:**

- Fan out: launch specialists in parallel

- Fan in: merge all outputs with a synthesis agent
<br>
<br>

**Always handle failures:**

- Default `gather()` raises on first exception — you lose all results

- `gather(..., return_exceptions=True)` returns exceptions as values

- Check each result: `isinstance(result, Exception)`

- Decide: enough good results to continue, or abort?
<br>
<br>

**Timeouts prevent runaway tasks:**

- Use `asyncio.wait_for()` to add timeouts — decide whether to skip, use a default, or abort

---

## 📍 Next Step

**Notebook 17: Debate & Critique Pattern**  

Put agents in dialogue with each other to improve output quality through internal challenge and refinement.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-16-parallel-execution)

---